## Math_Func_generator

In [28]:
import sympy as sp
import random
from joblib import Parallel, delayed
import json
import pandas as pd
from tqdm import tqdm

x = sp.symbols('x')

In [29]:
def base_functions(x):
    return [
        x,
        x**2,
        x**3,
        sp.sin(x),
        sp.cos(x),
        sp.exp(x),
        sp.log(1 + x),       
        sp.sqrt(1 + x**2)    
    ]

In [30]:
def rand_coeff():
    choices = [-3, -2, -1, -0.5, 0.5, 1, 2, 3]
    return random.choice(choices)

In [31]:
def random_transform(expr):
    transforms = [
        lambda e: sp.sin(e),
        lambda e: sp.cos(e),
        lambda e: sp.exp(e),
        lambda e: sp.log(1 + e**2),
        lambda e: e**2
    ]
    if random.random() < 0.4:
        return random.choice(transforms)(expr)
    return expr

In [32]:
def generate_expr(depth=0, max_depth=3):
    if depth >= max_depth:
        f = random.choice(base_functions(x))
        return rand_coeff() * f

    if random.random() < 0.4:
        f = random.choice(base_functions(x))
        expr = rand_coeff() * f
        return random_transform(expr)

    left = generate_expr(depth + 1, max_depth)
    right = generate_expr(depth + 1, max_depth)

    op = random.choice(['+', '*'])

    if op == '+':
        expr = left + right
    else:
        expr = left * right

    expr = random_transform(expr)
    return expr

In [33]:
def is_valid(expr):
    try:
        t = sp.series(expr, x, 0, 5).removeO()
        
        if not t.has(x):
            return False
        
        return True
    except:
        return False

In [34]:
def generate_one(_):
    try:
        expr = generate_expr(max_depth=1)

        expr = sp.expand(expr)

        if not is_valid(expr):
            return None

        taylor = sp.series(expr, x, 0, 5).removeO()
        taylor = sp.simplify(taylor)

        return (str(expr), str(taylor))

    except:
        return None

In [35]:
def generate_dataset_joblib(n_samples=5000, save_path="dataset.jsonl"):
    results = Parallel(n_jobs=-1, verbose=10)(
        delayed(generate_one)(i) for i in range(n_samples * 2)
    )

    data = [r for r in results if r is not None]

    return data


In [36]:
dataset = generate_dataset_joblib(5000)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    2.0s
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    2.1s
[Parallel(n_jobs=-1)]: Done  29 tasks      | elapsed:    2.3s
[Parallel(n_jobs=-1)]: Done  40 tasks      | elapsed:    2.5s
[Parallel(n_jobs=-1)]: Done  53 tasks      | elapsed:    2.7s
[Parallel(n_jobs=-1)]: Done  66 tasks      | elapsed:    2.8s
[Parallel(n_jobs=-1)]: Done  81 tasks      | elapsed:    3.0s
[Parallel(n_jobs=-1)]: Done  96 tasks      | elapsed:    3.2s
[Parallel(n_jobs=-1)]: Done 113 tasks      | elapsed:    3.3s
[Parallel(n_jobs=-1)]: Done 130 tasks      | elapsed:    3.4s
[Parallel(n_jobs=-1)]: Done 149 tasks      | elapsed:    3.6s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.19408417424535523s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done 168 tasks      | elapsed:    3.8s
[Parallel(n_jobs=-1)]: Done 191 tasks      | elapsed:    4.1s
[Parallel(n_jobs=-1)]

In [39]:
dataframe=pd.DataFrame(dataset,columns=["Function","Taylor Expression"])
dataframe

,Function,Taylor Expression
0,-6*x**2*log(x + 1),3*x**3*(x - 2)
1,x,x
2,-3*x**2 - 3,-3*x**2 - 3
3,-3*sqrt(x**2 + 1)*sin(x),x*(-x**2 - 3)
4,cos(log(x + 1)),-5*x**4/12 + x**3/2 - x**2/2 + 1
...,...,...
9400,-0.5*log(x + 1),x*(0.125*x**3 - 0.166666666666667*x**2 + 0.25*...
9401,-cos(x),-x**4/24 + x**2/2 - 1
9402,log(9*x**4 + 1),9*x**4
9403,3*x**2 + 2*log(x + 1),x*(-3*x**3 + 4*x**2 + 12*x + 12)/6
